# Run the Health Information Assistant on Colab

The demo needs a CUDA GPU: the INT8/INT4 checkpoints are BitsAndBytes
quantized, and BitsAndBytes only runs on CUDA. This notebook launches the
app from `webapp/` on a Colab GPU runtime and gives you a public share link.

## Before you start

1. **Runtime > Change runtime type** and pick a GPU:
   - **A100 (40 GB)** — same GPU class the benchmark ran on, and the only
     option that fits every configuration including the Gemma 4 12B baseline
     (24.98 GiB). Requires Colab Pro/Pro+.
   - **L4 (22 GB)** — fits every INT4/INT8 configuration and two baselines.
   - **T4 (15 GB)** — free tier; fits every INT4 and INT8 configuration.

   Cell 1 checks the GPU you actually got and prints which configurations
   fit, so you do not have to work this out by hand.
2. Add your Hugging Face token as a Colab secret named `HF_TOKEN`
   (key icon in the left sidebar, then enable *Notebook access*).
3. Accept the licence on all three gated model pages with the same account:
   - <https://huggingface.co/google/gemma-4-E4B-it>
   - <https://huggingface.co/google/gemma-4-12B-it>
   - <https://huggingface.co/google/medgemma-1.5-4b-it>

Then run the cells in order.

**Research demo only.** It does not provide medical diagnosis and does not
replace a qualified healthcare professional.

## 1. Check the GPU

Stop here if this reports no GPU: change the runtime type first.

In [1]:
# Diagnostics only. This cell never raises, so it cannot take the kernel
# down in frontends that treat SystemExit as a process exit.
torch = None
try:
    import torch
except Exception as exc:
    print("PyTorch failed to import:", f"{type(exc).__name__}: {exc}")
    print()
    print("If this is an INTERNAL ASSERT / tp_dict error, a pip install replaced")
    print("a library torch links against while the kernel was running.")
    print("Fix: restart the kernel or session, then run this cell first.")

if torch is None:
    pass
elif not torch.cuda.is_available():
    print("No CUDA GPU visible to PyTorch.")
    print(f"PyTorch build : {torch.__version__}")
    print()
    print("On Colab      : Runtime > Change runtime type, pick a GPU, rerun.")
    print("Running local : this notebook targets Colab. The assistant tab needs")
    print("                a CUDA GPU with about 4.3 GiB free for the smallest")
    print("                configuration. The Benchmark Explorer tab needs none:")
    print("                run  python webapp/app.py  instead.")
else:
    props = torch.cuda.get_device_properties(0)
    total_gib = props.total_memory / 2**30
    print(f"GPU            : {props.name}")
    print(f"VRAM           : {total_gib:.2f} GiB")
    print(f"Compute cap.   : {props.major}.{props.minor}")
    print(f"PyTorch        : {torch.__version__}")

    # Peak VRAM measured by the benchmark, smallest first.
    measured = [
        ("MedGemma 1.5 4B - INT4", 4.30),
        ("MedGemma 1.5 4B - INT8", 5.95),
        ("MedGemma 1.5 4B - Baseline BF16", 9.30),
        ("Gemma 4 12B - INT4", 9.85),
        ("Gemma 4 E4B - INT4", 10.80),
        ("Gemma 4 E4B - INT8", 12.80),
        ("Gemma 4 12B - INT8", 14.79),
        ("Gemma 4 E4B - Baseline BF16", 16.92),
        ("Gemma 4 12B - Baseline BF16", 24.98),
    ]
    # Leave headroom for the KV cache and activations.
    budget = total_gib - 1.5
    print()
    print("Configurations that should fit here:")
    for label, vram in measured:
        verdict = "yes" if vram <= budget else "no "
        print(f"  {verdict:<4} {label:<32} {vram:>6.2f} GiB")

    if props.major < 7:
        print()
        print("Warning: bitsandbytes targets compute capability 7.5+; this is older.")

    # The benchmark ran on an A100-SXM4-40GB. Matching the GPU makes the
    # hardware comparable, but the demo still measures a different task
    # (open-ended health questions) than the benchmark (constrained
    # multiple-choice scoring), so the numbers are still not one result.
    if "A100" in props.name:
        print()
        print("Same GPU class the benchmark ran on (A100-SXM4-40GB).")
        print("Hardware matches, but the demo measures a different task, so live")
        print("metrics are still not a reproduction of the benchmark numbers.")

GPU            : NVIDIA A100-SXM4-40GB
VRAM           : 39.49 GiB
Compute cap.   : 8.0
PyTorch        : 2.11.0+cu128

Configurations that should fit here:
  yes  MedGemma 1.5 4B - INT4             4.30 GiB
  yes  MedGemma 1.5 4B - INT8             5.95 GiB
  yes  MedGemma 1.5 4B - Baseline BF16    9.30 GiB
  yes  Gemma 4 12B - INT4                 9.85 GiB
  yes  Gemma 4 E4B - INT4                10.80 GiB
  yes  Gemma 4 E4B - INT8                12.80 GiB
  yes  Gemma 4 12B - INT8                14.79 GiB
  yes  Gemma 4 E4B - Baseline BF16       16.92 GiB
  yes  Gemma 4 12B - Baseline BF16       24.98 GiB

Same GPU class the benchmark ran on (A100-SXM4-40GB).
Hardware matches, but the demo measures a different task, so live
metrics are still not a reproduction of the benchmark numbers.


## 2. Get the code

If the repository is private, replace the URL with a token URL
(`https://<github-username>:<github-PAT>@github.com/peempat/medpubcodex.git`)
or upload the `webapp/` folder to `/content/medpubcodex/` by hand.

In [6]:
REPO_URL = "https://github.com/peempat/medpubcodex.git"
BRANCH = "feature/webapp"
TARGET = "/content/medpubcodex"

import subprocess
from pathlib import Path

if Path(TARGET).exists():
    print("Already cloned; pulling the latest commit.")
    subprocess.run(["git", "-C", TARGET, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", TARGET, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", TARGET, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, TARGET],
        check=True,
    )

print("\nwebapp/ contents:")
for path in sorted(Path(TARGET, "webapp").iterdir()):
    print("  ", path.name)

Already cloned; pulling the latest commit.

webapp/ contents:
   .env.example
   .gitignore
   README.md
   __pycache__
   app.py
   benchmark_results.py
   colab_launch.ipynb
   export_results.py
   inference.py
   model_registry.py
   prompts.py
   requirements.txt
   results
   safety.py
   tests


## 3. Install dependencies

Colab already ships a CUDA build of PyTorch, so torch is deliberately left
alone here. The other pins match the benchmark environment.

### Expected: a red pip dependency-conflict block

pip will report conflicts against packages Colab preinstalls but this project
never imports - `hf-gradio`, `google-genai`, `python-fasthtml`, `google-adk`.
Every version it complains about is the version `gradio` itself requires:

| pip complains about | what gradio 5.50 requires |
| --- | --- |
| `gradio-client 1.14.0` | `gradio-client==1.14.0` (exact pin) |
| `pydantic 2.12.3` | `pydantic<=2.12.3` |
| `starlette 0.52.1` | `starlette<1.0,>=0.40.0` |

Silencing those warnings would mean downgrading gradio and breaking the app.
Ignore them, and trust the verification the cell prints instead: it is green
when `torch in this kernel` and `torch on disk` both report a working CUDA
build. Restart the session only if the cell tells you to.

In [7]:
# Colab already ships a CUDA build of torch and a matching numpy.
# Nothing here upgrades either: only-if-needed keeps pip from pulling a
# newer numpy that would break the already-imported torch C extension.
%pip install -q --upgrade --upgrade-strategy only-if-needed \
  "gradio>=5.0,<6" "transformers==5.14.1" "bitsandbytes==0.49.2" \
  "accelerate>=1.14" "huggingface_hub>=1.2,<2" "sentencepiece>=0.2,<0.3" \
  "psutil>=5.9" "python-dotenv>=1.0"

import importlib
import subprocess
import sys

for name in ("gradio", "transformers", "bitsandbytes", "accelerate", "pandas"):
    try:
        module = importlib.import_module(name)
        print(f"{name:<16} {getattr(module, '__version__', 'unknown')}")
    except Exception as exc:
        print(f"{name:<16} FAILED: {type(exc).__name__}: {exc}")

# Two separate things can break, so check both.
# 1. The torch already loaded in THIS kernel - the one the app will use.
try:
    import torch

    torch.zeros(1).sum()
    print("torch in this kernel:", torch.__version__, "| working")
except Exception as exc:
    print("torch in THIS KERNEL is broken:", f"{type(exc).__name__}: {exc}")
    print("ACTION REQUIRED: Runtime > Restart session, then rerun from cell 1.")

# 2. The torch on disk, as a fresh interpreter sees it.
probe = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True,
    text=True,
)
print()
if probe.returncode == 0:
    print("torch on disk       :", probe.stdout.strip())
    print("Both checks passed. Continue to the next cell.")
else:
    print("torch is broken after the install:")
    print(probe.stderr.strip()[-600:])
    print()
    print("ACTION REQUIRED: Runtime > Restart session, then rerun from cell 1.")

gradio           5.50.0
transformers     5.14.1
bitsandbytes     0.49.2
accelerate       1.14.0
pandas           2.2.3
torch in this kernel: 2.11.0+cu128 | working

torch on disk       : 2.11.0+cu128 True
Both checks passed. Continue to the next cell.


## 4. Hugging Face token

Read from Colab secrets so the token is never typed into a cell and never
saved in the notebook output.

In [8]:
# Reads the token from a Colab secret, falling back to the HF_TOKEN
# environment variable so this also works outside Colab. Never raises.
import os

token = None
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except ImportError:
    token = os.environ.get("HF_TOKEN")
    if token:
        print("Not running on Colab; using the HF_TOKEN environment variable.")
except Exception as exc:
    print("Could not read the Colab secret HF_TOKEN:", f"{type(exc).__name__}: {exc}")

if not token:
    print("No Hugging Face token available.")
    print()
    print("On Colab : open the key icon in the left sidebar, add a secret named")
    print("           HF_TOKEN, and switch on Notebook access.")
    print("Local    : set the HF_TOKEN environment variable, or fill in")
    print("           webapp/.env (see webapp/.env.example).")
else:
    os.environ["HF_TOKEN"] = token

    from huggingface_hub import HfApi

    api = HfApi(token=token)
    try:
        print("Logged in as:", api.whoami()["name"])
    except Exception as exc:
        print("Token rejected by Hugging Face:", f"{type(exc).__name__}: {exc}")

    REPOS = [
        "google/medgemma-1.5-4b-it",
        "google/gemma-4-E4B-it",
        "google/gemma-4-12B-it",
        "pupupapapa/medgemma-1.5-4b-it-int4-bnb",
        "pupupapapa/medgemma-1.5-4b-it-int8-bnb",
        "pupupapapa/gemma-4-e4b-it-int4-bnb",
        "pupupapapa/gemma-4-e4b-it-int8-bnb",
        "pupupapapa/gemma-4-12b-it-int4-bnb",
        "pupupapapa/gemma-4-12b-it-int8-bnb",
    ]
    print()
    print("Access check:")
    for repo in REPOS:
        try:
            api.model_info(repo)
            print(f"  ok       {repo}")
        except Exception as exc:
            print(f"  BLOCKED  {repo}  ({type(exc).__name__})")
    print()
    print("A BLOCKED google/* repo means the licence has not been accepted yet.")

Could not read the Colab secret HF_TOKEN: TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.
No Hugging Face token available.

On Colab : open the key icon in the left sidebar, add a secret named
           HF_TOKEN, and switch on Notebook access.
Local    : set the HF_TOKEN environment variable, or fill in
           webapp/.env (see webapp/.env.example).


## 5. Launch

This prints a `*.gradio.live` public link, valid for 72 hours. The first
question downloads the checkpoint, so it takes a few minutes; later questions
reuse the loaded model.

Stop the cell to shut the app down.

In [5]:
import sys

WEBAPP = "/content/medpubcodex/webapp"
if WEBAPP not in sys.path:
    sys.path.insert(0, WEBAPP)

import app

app.build_interface().launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6c25071a890c190e84.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Optional: run the tests

None of these load a model, so they finish in seconds.

In [ ]:
!cd /content/medpubcodex && python -m pytest webapp/tests -q

## Troubleshooting

### `RuntimeError: THPDtypeType.tp_dict == nullptr` on `import torch`

The torch C extension loaded in this kernel no longer matches what is on
disk. It is almost always caused by a pip install replacing one of torch's
dependencies while the kernel was running, or by running the cells out of
order.

1. **Runtime > Restart session** (`Ctrl+M .`), then run the cells in order.
2. Still broken: **Runtime > Disconnect and delete runtime**, reconnect,
   and run 1 -> 2 -> 3 -> restart -> 4 -> 5.
3. Still broken: run the recovery cell below, then restart the session.

### `403` when loading a model

The licence has not been accepted for that repository. Cell 4 lists exactly
which repositories your token cannot reach.

### `CUDA out of memory`

Pick a configuration that cell 1 marked `yes`, or use *Unload model / free
GPU memory* in the app before switching.

### Recovery: reinstall torch to match this runtime

Only run this if a restart did not fix the import. It reinstalls the torch
build Colab shipped, then you must restart the session.

In [ ]:
import subprocess
import sys

print(subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir", "torch"],
    capture_output=True,
    text=True,
).stdout[-1500:])
print()
print("Now do: Runtime > Restart session, then rerun from cell 1.")